# CARE · Entraînement du modèle « objets oubliés » (Colab, GPU gratuit)

Entraîne un **YOLO26s** sur la version Roboflow `care-oubli` (classes `baby`, `keys`, `wallet`, `card`)
avec le script du dépôt (`entrainement/entrainer.py`), puis récupère `care-v1.pt` à déposer dans
`modeles/` du banc.

**Avant de lancer :** menu *Exécution → Modifier le type d'exécution → GPU (T4)*.
Durée indicative : 20 à 40 min pour ~2 500 images à 640 px.

In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv || echo "Pas de GPU : Exécution > Modifier le type d'exécution > T4"

## 1. Code du banc et dépendances

In [ ]:
!git clone -q -b detection-surveillance https://github.com/auroremonnet/Stellantis-d-tection-oubli.git care
%cd care
!pip install -q "ultralytics>=8.4" roboflow

## 2. Clé API Roboflow

Roboflow → *Settings → API Keys*. La clé est saisie ici, jamais écrite dans le code ni dans le dépôt.

In [ ]:
import os, getpass
os.environ["ROBOFLOW_API_KEY"] = getpass.getpass("Clé API Roboflow : ")

## 3. Entraînement

Réglages par défaut (guide de fine-tuning Ultralytics, petit jeu proche de COCO) : 10 couches gelées,
AdamW lr0 = 0,001, arrêt précoce après 20 epochs sans progrès. Pour les petits objets (clés, carte),
essayer ensuite `--imgsz 1280` et comparer le rappel.

In [ ]:
VERSION = 1   # version du jeu Roboflow care-oubli
!python entrainement/entrainer.py --version {VERSION} --modele yolo26s.pt --epochs 100 --imgsz 640 --batch -1 --nom care-v{VERSION}

## 4. Résultats

In [ ]:
import json
from IPython.display import Image, display
fiche = json.load(open(f"modeles/care-v{VERSION}.json"))
print(json.dumps(fiche["metriques"], indent=2, ensure_ascii=False))
run = fiche["run"]
for img in ("results.png", "confusion_matrix_normalized.png", "BoxPR_curve.png"):
    p = f"{run}/{img}"
    if os.path.exists(p):
        display(Image(p, width=900))

## 5. Évaluation comparative (facultatif)

Compare YOLOE seul, le modèle entraîné seul et l'hybride sur le split **test** (jamais vu à
l'entraînement). Produit un tableau Markdown prêt pour le rapport dans `rapports/`.

In [ ]:
!pip install -q "git+https://github.com/ultralytics/CLIP.git"
!python outils/evaluer.py --donnees datasets/care-oubli-v{VERSION}/data.yaml --split test --moteurs yoloe custom hybride
import glob; print(open(sorted(glob.glob("rapports/*.md"))[-1]).read())

## 6. Récupérer le modèle

Copier `care-v1.pt` dans le dossier `modeles/` du banc : le moteur `hybride` le charge au démarrage.

In [ ]:
from google.colab import files
files.download(f"modeles/care-v{VERSION}.pt")
files.download(f"modeles/care-v{VERSION}.json")